# Projekt 3: Analiza Semantyczna (Semantic Analysis)

Cel: Analiza semantyczna korpusu języka niemieckiego (100 tys. zdań).
Zadania:
1. Budowa grafu dwudzielnego łączącego przymiotniki z rzeczownikami.
2. Budowa grafu dwudzielnego łączącego czasowniki z rzeczownikami.
3. Wyświetlenie list połączeń dla top 100 słów z każdej kategorii.
4. Wizualizacja ze skalą kolorów opartą na częstości występowania.

In [ ]:
import os
import spacy
from collections import Counter, defaultdict
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm

# Ensure plots display inline
%matplotlib inline

## 1. Konfiguracja i Ładowanie Danych

Ładujemy model języka niemieckiego spaCy oraz plik z danymi.

In [ ]:
# Configuration
DATA_DIR = "../../data"
EXTRACTED_DIR = "deu_mixed-typical_2011_100K"
EXTRACTED_FILE_NAME = "deu_mixed-typical_2011_100K-sentences.txt"
FILE_PATH = os.path.join(DATA_DIR, EXTRACTED_DIR, EXTRACTED_FILE_NAME)

# Load spaCy model
try:
    nlp = spacy.load("de_core_news_sm")
except OSError:
    print("Downloading spaCy model...")
    !python -m spacy download de_core_news_sm
    nlp = spacy.load("de_core_news_sm")

## 2. Przetwarzanie i Ekstrakcja Relacji

Iterujemy przez zdania i ekstrahujemy:
- Pary (Przymiotnik, Rzeczownik)
- Pary (Czasownik, Rzeczownik)
- Liczniki wystąpień dla każdego słowa (do wyboru top 100)

In [ ]:
noun_counts = Counter()
adj_counts = Counter()
verb_counts = Counter()

adj_noun_pairs = []
verb_noun_pairs = []

print(f"Processing {FILE_PATH}...")

limit = 100000  # Process all or limit for testing

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for i, line in tqdm(enumerate(f), total=limit):
        if i >= limit:
            break
        
        parts = line.split('\t', 1)
        if len(parts) != 2:
            continue
            
        sentence = parts[1]
        doc = nlp(sentence)
        
        for token in doc:
            # Count POS frequencies
            if token.pos_ == "NOUN":
                noun_counts[token.lemma_.lower()] += 1
            elif token.pos_ == "ADJ":
                adj_counts[token.lemma_.lower()] += 1
            elif token.pos_ == "VERB":
                verb_counts[token.lemma_.lower()] += 1
            
            # Extract Adj-Noun relations (Adjective modifies Noun)
            # Looking for tokens that are ADJ and have a NOUN head
            if token.pos_ == "ADJ" and token.head.pos_ == "NOUN":
                adj_noun_pairs.append((token.lemma_.lower(), token.head.lemma_.lower()))
            
            # Extract Verb-Noun relations (Noun is Subject or Object of Verb)
            # Case 1: Token is NOUN and head is VERB
            if token.pos_ == "NOUN" and token.head.pos_ == "VERB":
                verb_noun_pairs.append((token.head.lemma_.lower(), token.lemma_.lower()))
            
            # Case 2: Token is VERB and has NOUN children (less common to traverse this way but good for coverage)
            # We stick to the dependency child -> head direction for simplicity as it covers the relation.

## 3. Wybór Top 100 Słów

Wybieramy 100 najczęstszych rzeczowników, przymiotników i czasowników.

In [ ]:
top_nouns = set(word for word, count in noun_counts.most_common(100))
top_adjs = set(word for word, count in adj_counts.most_common(100))
top_verbs = set(word for word, count in verb_counts.most_common(100))

print(f"Top Nouns example: {list(top_nouns)[:5]}")
print(f"Top Adjs example: {list(top_adjs)[:5]}")
print(f"Top Verbs example: {list(top_verbs)[:5]}")

## 4. Analiza Przymiotnik - Rzeczownik

Filtrujemy pary, aby zachować tylko te, które zawierają słowa z naszych list top 100.

In [ ]:
# Filter pairs
filtered_adj_noun = [
    (adj, noun) 
    for adj, noun in adj_noun_pairs 
    if adj in top_adjs and noun in top_nouns
]

# Count co-occurrences
adj_noun_counts = Counter(filtered_adj_noun)

# Group by Adjective for listing
adj_to_nouns = defaultdict(list)
for (adj, noun), count in adj_noun_counts.items():
    adj_to_nouns[adj].append((noun, count))

# Sort nouns by count for each adjective
for adj in adj_to_nouns:
    adj_to_nouns[adj].sort(key=lambda x: x[1], reverse=True)

# Function to format the list as requested
def print_formatted_list(data_dict, category_name):
    print(f"--- Lista dla {category_name} ---")
    for head_word, items in sorted(data_dict.items()):
        items_str = ", ".join([f"{word}[{count}]" for word, count in items])
        print(f"Lista-dla_{head_word}: ({items_str})")

print_formatted_list(adj_to_nouns, "Przymiotnik")

### Wizualizacja: Graf Przymiotnik-Rzeczownik

In [ ]:
def get_color(weight):
    if weight == 0:
        return 'red'
    elif weight == 1:
        return 'yellow'
    elif weight > 10:
        return 'blue'
    elif weight > 2:
        return 'green'
    else:
        return 'yellow' # Fallback for 2

def draw_bipartite_graph(pairs_counts, set1_nodes, set2_nodes, title):
    B = nx.Graph()
    B.add_nodes_from(set1_nodes, bipartite=0)
    B.add_nodes_from(set2_nodes, bipartite=1)
    
    edges = []
    colors = []
    weights = []
    
    for (u, v), weight in pairs_counts.items():
        B.add_edge(u, v, weight=weight)
        edges.append((u, v))
        colors.append(get_color(weight))
        weights.append(weight * 0.5) # Scale width
        
    pos = nx.bipartite_layout(B, set1_nodes)
    
    plt.figure(figsize=(15, 20))
    nx.draw_networkx_nodes(B, pos, nodelist=set1_nodes, node_color='lightblue', node_size=500, alpha=0.8)
    nx.draw_networkx_nodes(B, pos, nodelist=set2_nodes, node_color='lightgreen', node_size=500, alpha=0.8)
    
    nx.draw_networkx_edges(B, pos, edgelist=edges, edge_color=colors, width=1.0, alpha=0.6)
    nx.draw_networkx_labels(B, pos, font_size=8)
    
    plt.title(title)
    plt.axis('off')
    plt.show()

# Visualize a subset to keep it readable (e.g., top 20 of each instead of 100, or filter by weight)
# If we plot 100x100 it will be very messy. Let's filter for edges with weight > 1 for clearer visualization
# or just plot the whole thing if required.

top_adj_noun_counts = {k: v for k, v in adj_noun_counts.items() if v >= 1}
active_adjs = set(k[0] for k in top_adj_noun_counts.keys())
active_nouns = set(k[1] for k in top_adj_noun_counts.keys())

draw_bipartite_graph(top_adj_noun_counts, list(active_adjs), list(active_nouns), "Graf Przymiotnik-Rzeczownik (Kolory krawędzi oznaczają siłę związku)")

## 5. Analiza Czasownik - Rzeczownik

In [ ]:
# Filter pairs
filtered_verb_noun = [
    (verb, noun) 
    for verb, noun in verb_noun_pairs 
    if verb in top_verbs and noun in top_nouns
]

# Count co-occurrences
verb_noun_counts = Counter(filtered_verb_noun)

# Group by Verb for listing
verb_to_nouns = defaultdict(list)
for (verb, noun), count in verb_noun_counts.items():
    verb_to_nouns[verb].append((noun, count))

# Sort
for verb in verb_to_nouns:
    verb_to_nouns[verb].sort(key=lambda x: x[1], reverse=True)

# Print List
print_formatted_list(verb_to_nouns, "Czasownik")

# Visualize
top_verb_noun_counts = {k: v for k, v in verb_noun_counts.items() if v >= 1}
active_verbs = set(k[0] for k in top_verb_noun_counts.keys())
active_nouns_v = set(k[1] for k in top_verb_noun_counts.keys())

draw_bipartite_graph(top_verb_noun_counts, list(active_verbs), list(active_nouns_v), "Graf Czasownik-Rzeczownik")